In [0]:
import dlt
import re
import unicodedata

from pyspark.sql.functions import col, regexp_extract
from pyspark.sql.types import *

# ==========================================
# CONFIGS
# ==========================================
SOURCE_PATH = "/Volumes/..."
SCHEMA_LOCATION = "/Volumes/..."

# ==========================================
# SCHEMA FIXO (RECEITA FEDERAL - EMPRESAS)
# ==========================================
schema_empresas = StructType([
    StructField("cnpj_basico", StringType(), True),
    StructField("razao_social", StringType(), True),
    StructField("natureza_juridica", StringType(), True),
    StructField("qualificacao_responsavel", StringType(), True),
    StructField("capital_social", StringType(), True),
    StructField("porte_empresa", StringType(), True),
    StructField("ente_federativo_responsavel", StringType(), True)
])

# ==========================================
# FUNÇÕES UTILITÁRIAS
# ==========================================

def clean_column_name(col_name: str) -> str:
    col_name = (
        unicodedata.normalize("NFKD", col_name)
        .encode("ascii", "ignore")
        .decode("utf-8")
    )

    col_name = col_name.lower()
    col_name = re.sub(r"[^a-z0-9]", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name)

    return col_name.strip("_")


def normalize_columns(df):

    new_cols = []
    seen = {}

    for c in df.columns:

        new_name = clean_column_name(c)

        if new_name in seen:
            seen[new_name] += 1
            new_name = f"{new_name}_{seen[new_name]}"
        else:
            seen[new_name] = 0

        new_cols.append(new_name)

    return df.toDF(*new_cols)


# ==========================================
# BRONZE TABLE
# ==========================================

@dlt.table(
    name="lab.bronze.companies",
    comment="Ingestão Receita Federal - Empresas",
    table_properties={
        "layer": "bronze"
    },
    partition_cols=["ano_mes"]
)
def bronze_empresas():

    df = (
        spark.readStream
        .format("cloudFiles")

        # =========================
        # AUTO LOADER
        # =========================
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("cloudFiles.inferColumnTypes", "false")

        # =========================
        # CSV
        # =========================
        .option("header", "false")
        .option("delimiter", ";")
        .option("encoding", "UTF-8")
        .option("ignoreLeadingWhiteSpace", "true")
        .option("ignoreTrailingWhiteSpace", "true")
        .option("multiLine", "false")

        # =========================
        # SCHEMA FIXO
        # =========================
        .schema(schema_empresas)

        .load(SOURCE_PATH)
    )

    # ==========================================
    # NORMALIZAÇÃO
    # ==========================================
    df = normalize_columns(df)

    # ==========================================
    # COMPETÊNCIA DA CARGA
    # ==========================================
    df = (
        df.withColumn(
            "ano_mes",
            regexp_extract(
                col("_metadata.file_path"),
                r"_(\d{6})\.csv$",
                1
            ).cast("int")
        )
    )

    return df